# Session 2 Demo: Problem Formulation
**How do we turn a real-world problem into a search problem?**


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Visualization library  ─  run this cell first
# ─────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))
import fai_viz
print("✅ fai_viz visualization library loaded")

## Section 1: The 4 Components of a Search Problem

- **State***: a snapshot of the world at one moment
- **Initial State**: where we start
- **Actions** (Successor Function): what moves are possible from a state
- **Goal Test**: how we know we're done
- **Path Cost**: cost of a sequence of actions

## Section 2: Example 1 — Grid Navigation

A robot must navigate from (0,0) to (4,4) on a 5×5 grid.

### 📌 Problem Reference: Grid Navigation

Below is the **5×5 grid** used throughout this session. The robot starts at **S (0,0)** and must reach **G (4,4)**. Run this cell to see the layout clearly.

*(Right panel shows the grid with a wall blocking column 2 — used in the wall example.)*

In [ ]:
# Show the Grid Navigation reference figure
fai_viz.show_grid_figure()

In [ ]:
# ── Problem: Grid Navigation ─────────────────────────────────
# State      : (row, col)  — position of the robot
# Initial    : (0, 0)
# Goal       : (4, 4)
# Actions    : up / down / left / right
# Path cost  : 1 per step

def make_grid_problem(initial, goal, grid_size=5, walls=None): 
    return {"initial": initial, "goal": goal,
            "grid_size": grid_size, "walls": walls or set()}  # dictionary to hold problem parameters
def is_goal(problem, state):
    return state == problem["goal"]

def get_actions(problem, state):
    """Return [(action_name, next_state), ...] for each valid move."""
    row, col = state # unpack current position
    size = problem["grid_size"]
    walls = problem["walls"]
    candidates = [("up",    (row-1, col)),
                  ("down",  (row+1, col)),
                  ("left",  (row,   col-1)),
                  ("right", (row,   col+1))]
    return [(name, ns) for name, ns in candidates
            if 0 <= ns[0] < size and 0 <= ns[1] < size and ns not in walls]

def action_cost(problem, state, action):
    return 1   # uniform grid

In [ ]:
# Test
p = make_grid_problem((0,0), (4,4)) # create a grid problem with initial state (0,0) and goal state (4,4)
print("Initial state:", p["initial"])
print("Goal state   :", p["goal"])
print("Is (4,4) goal?", is_goal(p, (4,4)))
print("Actions from (0,0):", get_actions(p, (0,0)))  # 2 moves (corner)
print("Actions from (2,2):", get_actions(p, (2,2)))  # 4 moves

In [ ]:
# Add a wall: column 2 is blocked (rows 0-3) → forces detour
walls = {(0,2),(1,2),(2,2),(3,2)}
p_wall = make_grid_problem((0,0), (4,4), walls=walls)
print("Actions from (1,1) with wall:", get_actions(p_wall, (1,1)))
# Note: (1,2) is blocked → no 'right' move from (1,1)

## Section 3: Example 2 — Water Pouring Problem

Three buckets: 3L, 5L, 9L. Goal: measure exactly 7L in any bucket.

| a | b | c | Comment |
|---|---|---|---------|
| 0 | 0 | 0 | start |
| 3 | 0 | 0 | |
| 0 | 0 | 3 | |
| ... | ... | ... | |
| 0 | 5 | 7 | goal! |

### 📌 Problem Reference: Water Pouring Problem

Three jugs: **3L, 5L, 9L** — all start empty. Goal: measure exactly **7 litres** in any jug. Run this cell to see the setup.

Operations: **Fill** any jug to its capacity | **Empty** any jug | **Pour** from one jug into another (until source empty or target full).

In [ ]:
# Show the Water Pouring reference figure (3-jug demo version)
fai_viz.show_water_jug_figure(capacities=(3, 5, 9), goal_litres=7)

In [ ]:
# ── Problem: Water Pouring ───────────────────────────────────
# State      : (level_a, level_b, level_c) — current water levels
# Initial    : (0, 0, 0)
# Goal       : any bucket contains exactly 7L
# Actions    : Fill i, Empty i, Pour i→j
# Path cost  : 1 per action

def make_pour_problem(initial, goal, capacities):
    return {"initial": initial, "goal": goal, "capacities": capacities}

def pour_is_goal(problem, state):
    return problem["goal"] in state   # e.g. 7 in (0, 5, 7) → True

def pour_get_actions(problem, state):
    caps = problem["capacities"]        #  capacities of the buckets
    n = len(state)
    actions = []                        # list to hold possible actions and resulting states, e.g. [("Fill 0", (3,0,0)), ("Pour 0→1", (0,3,0)), ...]
    for i in range(n):
        if state[i] < caps[i]:          # can fill bucket i to capacity
            new = list(state)           # convert state tuple to list to modify it 
            new[i] = caps[i]           
            actions.append((f"Fill {i}", tuple(new))) # add the action and resulting state to the actions list
        if state[i] > 0:                # can empty bucket i
            new = list(state)
            new[i] = 0
            actions.append((f"Empty {i}", tuple(new)))
        for j in range(n):              # can pour i → j
            if i != j and state[i] > 0 and state[j] < caps[j]:
                new = list(state)
                amount = min(state[i], caps[j] - state[j]) 
                new[i] -= amount
                new[j] += amount
                actions.append((f"Pour {i}→{j}", tuple(new)))
    return actions

In [ ]:
pour_p = make_pour_problem((0,0,0), 7, (3,5,9))
print("Is (0,5,7) goal?", pour_is_goal(pour_p, (0,5,7)))   # True
print("Is (3,0,0) goal?", pour_is_goal(pour_p, (3,0,0)))   # False
print("\nActions from (0,2,5):")
for action, next_state in pour_get_actions(pour_p, (0,2,5)):
    print(f"  {action:10s} → {next_state}")

## Section 4: The expand() function — connecting state to search tree

In the slides, `Expand(node, Actions[p])` generates all child nodes. Here's the Python implementation:

In [ ]:
def make_node(state, parent=None, action=None, cost=0, depth=0): # helper function to create a search node with given parameters
    return {"state": state, "parent": parent, "action": action, "cost": cost, "depth": depth} # node is a dictionary with keys: 'state', 'parent', 'action', 'cost', 'depth'

def expand(problem, node, get_actions_fn, action_cost_fn): # helper function to expand a search node using the provided get_actions and action_cost functions
    """Expand a node → return list of child nodes."""
    children = []
    for action_name, next_state in get_actions_fn(problem, node["state"]): # get possible actions and resulting states from the current node's state
        step_cost = action_cost_fn(problem, node["state"], action_name)
        children.append(make_node(next_state, node, action_name, node["cost"] + step_cost, node["depth"] + 1))
    return children

# Demo: expand the root node for the grid problem
root = make_node((0,0))
children = expand(p, root, get_actions, action_cost)
print(f"Expanding (0,0) → {len(children)} children:")
for c in children:
    print(f"  action={c['action']:6s}  next_state={c['state']}") 

# Demo: expand the root node for the water pouring problem
pour_root = make_node((0,0,0))
pour_children = expand(pour_p, pour_root, pour_get_actions, action_cost)
print(f"\nExpanding (0,0,0) → {len(pour_children)} children:")
for c in pour_children:
    print(f"  action={c['action']:15s}  next_state={c['state']}")

## Section 5: Summary — Mapping to the Pseudocode

| Pseudocode Term | Python Code |
|---|---|
| `Initial-State[p]` | `problem["initial"]` |
| `Goal-Test[p](state)` | `is_goal(problem, state)` |
| `Actions[p]` | `get_actions(problem, state)` |
| `Expand(node, Actions[p])` | `expand(problem, node, get_actions, action_cost)` |
| `Make-Node(state)` | `make_node(state)` |

## 🎨 Visual: Grid Navigation State Space

The plot below shows the **grid problem** — start state (green), goal state (red), and the BFS solution path (blue arrows).

In [ ]:
from collections import deque

def bfs_demo(problem):
    """BFS — returns the solution path as a list of states."""
    start_node = make_node(problem["initial"])
    frontier = deque([start_node])
    visited = set([problem["initial"]])
    while frontier:
        node = frontier.popleft()
        if node["state"] == problem["goal"]:
            path = []
            cur = node
            while cur:
                path.append(cur["state"])
                cur = cur["parent"]
            return list(reversed(path))
        for _, ns in get_actions(problem, node["state"]):
            if ns not in visited:
                visited.add(ns)
                frontier.append(make_node(ns, node))
    return []

p_vis = make_grid_problem((0, 0), (4, 4), grid_size=5)
path_vis = bfs_demo(p_vis)
fai_viz.plot_grid_path(p_vis, path_vis, "Grid Navigation — BFS Optimal Path")
print(f"BFS path ({len(path_vis)-1} steps): {' → '.join(str(s) for s in path_vis)}")

## 🎨 Visual: Water Jug State Path

Each pair of bars shows the water level in each jug at each step of the BFS solution.

In [ ]:
from collections import deque

def bfs_jug(problem):
    """BFS for water jug — returns solution path as list of states."""
    start_node = make_node(problem["initial"])
    frontier = deque([start_node]) # first-in-first-out queue for BFS
    visited = {problem["initial"]} # set to keep track of visited states to avoid cycles
    while frontier:
        node = frontier.popleft()
        if pour_is_goal(problem, node["state"]):
            path = []
            cur = node
            while cur:
                path.append(cur["state"]) # build the path by following parent pointers back to the start node
                cur = cur["parent"] # move up to the parent node
            return list(reversed(path))
        for _, ns in pour_get_actions(problem, node["state"]): 
            if ns not in visited:
                visited.add(ns)
                frontier.append(make_node(ns, node)) # add the new state as a child node to the frontier 
    return []

pour_p2 = make_pour_problem((0, 0,0), 7, (3, 5,9))
jug_path = bfs_jug(pour_p2)
fai_viz.plot_jug_solution(jug_path, (3, 5,9), "Water Jug (3L, 5L, 9L) → Goal: 7L")
print(f"Solution in {len(jug_path)-1} steps: {jug_path}")